In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from tensorflow.examples.tutorials.mnist import input_data
mnist = input_data.read_data_sets('./mnist/data/', one_hot=True)

Extracting ./mnist/data/train-images-idx3-ubyte.gz
Extracting ./mnist/data/train-labels-idx1-ubyte.gz
Extracting ./mnist/data/t10k-images-idx3-ubyte.gz
Extracting ./mnist/data/t10k-labels-idx1-ubyte.gz


In [3]:
total_epoch = 100
batch_size = 100
learning_rate = 0.0002

n_input = 28 * 28
n_noise = 128
n_hidden = 256
n_class = 10

In [4]:
X = tf.placeholder(tf.float32, [None, n_input])
Y = tf.placeholder(tf.float32, [None, n_class])
Z = tf.placeholder(tf.float32, [None, n_noise])

In [5]:
def generator(noise, labels):
    with tf.variable_scope('generator'):
        inputs = tf.concat([noise, labels], 1)
        hidden = tf.layers.dense(inputs, n_hidden, activation=tf.nn.relu)
        output = tf.layers.dense(hidden, n_input, activation=tf.nn.sigmoid)
    
    return output

def discriminator(inputs, labels, reuse=None):
    with tf.variable_scope('discriminator') as scope:
        if reuse:
            scope.reuse_variables()
        inputs = tf.concat([inputs, labels], 1)
        hidden = tf.layers.dense(inputs, n_hidden, activation=tf.nn.relu)
        output = tf.layers.dense(hidden, 1, activation=None)
    
    return output

def get_noise(batch_size, n_noise):
    return np.random.uniform(-1., 1., size=(batch_size, n_noise))

In [6]:
G = generator(Z, Y)
D_real = discriminator(X, Y)
D_gene = discriminator(G, Y, True)

In [7]:
loss_D_real = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(logits=D_real, labels=tf.ones_like(D_real)))
loss_D_gene = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(logits=D_gene, labels=tf.zeros_like(D_gene)))
loss_D = loss_D_real + loss_D_gene

loss_G = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(logits=D_gene, labels=tf.ones_like(D_gene)))

D_var_list = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='discriminator')
G_var_list = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='generator')

train_D = tf.train.AdamOptimizer().minimize(loss_D, var_list=D_var_list)
train_G = tf.train.AdamOptimizer().minimize(loss_G, var_list=G_var_list)

In [8]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

total_batch = int(mnist.train.num_examples / batch_size)
loss_val_D, loss_val_G = 0, 0

for epoch in range(total_epoch):
    for i in range(total_batch):
        batch_xs, batch_ys = mnist.train.next_batch(batch_size)
        noise = get_noise(batch_size, n_noise)
        
        _, loss_val_D = sess.run([train_D, loss_D], feed_dict={X: batch_xs, Y: batch_ys, Z: noise})
        _, loss_val_G = sess.run([train_G, loss_G], feed_dict={Y: batch_ys, Z: noise})
    
    print('Epoch:', '%04d' % epoch, 'D loss: {:.4}'.format(loss_val_D), 'G loss: {:.4}'.format(loss_val_G))

    if epoch == 0 or (epoch + 1) % 10 == 0:
        sample_size = 10
        noise = get_noise(sample_size, n_noise)
        samples = sess.run(G, feed_dict={Y: mnist.test.labels[:sample_size], Z: noise})

        fig, ax = plt.subplots(2, sample_size, figsize=(sample_size, 2))

        for i in range(sample_size):
            ax[0][i].set_axis_off()
            ax[1][i].set_axis_off()

            ax[0][i].imshow(np.reshape(mnist.test.images[i], (28, 28)))
            ax[1][i].imshow(np.reshape(samples[i], (28, 28)))

        plt.savefig('samples2/{}.png'.format(str(epoch).zfill(3)), bbox_inches='tight')
        plt.close(fig)

print('Training Completed !')

Epoch: 0000 D loss: 0.006819 G loss: 7.385
Epoch: 0001 D loss: 0.04219 G loss: 8.34
Epoch: 0002 D loss: 0.004957 G loss: 7.149
Epoch: 0003 D loss: 0.04513 G loss: 5.961
Epoch: 0004 D loss: 0.05408 G loss: 6.633
Epoch: 0005 D loss: 0.1003 G loss: 6.448
Epoch: 0006 D loss: 0.09941 G loss: 7.169
Epoch: 0007 D loss: 0.08056 G loss: 10.15
Epoch: 0008 D loss: 0.14 G loss: 5.983
Epoch: 0009 D loss: 0.2556 G loss: 6.217
Epoch: 0010 D loss: 0.1632 G loss: 4.9
Epoch: 0011 D loss: 0.121 G loss: 5.225
Epoch: 0012 D loss: 0.2713 G loss: 3.511
Epoch: 0013 D loss: 0.2192 G loss: 4.427
Epoch: 0014 D loss: 0.479 G loss: 4.042
Epoch: 0015 D loss: 0.3876 G loss: 2.879
Epoch: 0016 D loss: 0.5091 G loss: 3.872
Epoch: 0017 D loss: 0.6086 G loss: 2.958
Epoch: 0018 D loss: 0.4124 G loss: 3.365
Epoch: 0019 D loss: 0.5967 G loss: 3.043
Epoch: 0020 D loss: 0.4872 G loss: 3.113
Epoch: 0021 D loss: 0.5527 G loss: 3.249
Epoch: 0022 D loss: 0.6526 G loss: 3.565
Epoch: 0023 D loss: 0.7828 G loss: 3.079
Epoch: 0024 D 